In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [3]:
with open('en_de.txt', 'r') as f:
  data = f.read()
  data = data.split('\n\n')[:-1]
  data = [x.split('\n') for x in data]
  en = [x[0] for x in data]
  de = [x[1] for x in data]

In [4]:
len(data)

297597

In [5]:
avg_count = [len(x[0].split()) for x in data]
sum(avg_count)/len(avg_count)

17.965423038538695

In [6]:
avg_count = [len(x[1].split()) for x in data]
sum(avg_count)/len(avg_count)

16.4192582586518

In [7]:
#en[:4]
#de[:4]
data[0]

["I've spent the last decade subjecting myself to pain and humiliation, hopefully for a good cause, which is self-improvement.",
 'Das letzte Jahrzehnt habe ich damit verbracht, mich Schmerz und Demütigung auszusetzen, hoffentlich für eine gute Sache, die Selbstvervollkommnung.']

In [8]:
import re
import html

def preprocess_text(text):

    text = html.unescape(text)
    url_pattern = r"https?://\S+|www\.\S+"
    text = re.sub(url_pattern, " <url> ", text)

    text = text.lower()

    text = re.sub(r"[‘’´`]", "'", text)
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[—–]', '-', text)

    text = text.replace("<url>", "PLACEHOLDERURL")

    text = re.sub(r"([!\"#$%&'()*+,-./:;<=>?@[\\\]^_`{|}~])", r" \1 ", text)

    text = text.replace("PLACEHOLDERURL", " <url> ")

    text = re.sub(r"\s+", " ", text).strip()

    return text


en = [preprocess_text(x) for x in en]
de = [preprocess_text(x) for x in de]

In [ ]:
de[:4]

NameError: name 'de' is not defined

In [9]:
from tokenizers import ByteLevelBPETokenizer
import os
os.makedirs("my_bpe_tokenizer_", exist_ok=True)

data = en + de
with open("corpus.txt", "w") as f:
    f.write("\n".join(data))

bpe_tokenizer = ByteLevelBPETokenizer()

bpe_tokenizer.train(
    files=["corpus.txt"],
    vocab_size=20000,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<url>"]
)

bpe_tokenizer.save_model("my_bpe_tokenizer_")
print("BPE Tokenizer successfully trained and saved!")

encoded = bpe_tokenizer.encode(en[5])
print("\nSample Original:", en[5])
print("Tokens:", encoded.tokens)
print("Token IDs:", encoded.ids)

BPE Tokenizer successfully trained and saved!

Sample Original: and this was an amazing year .
Tokens: ['and', 'Ġthis', 'Ġwas', 'Ġan', 'Ġamazing', 'Ġyear', 'Ġ.']
Token IDs: [348, 415, 393, 350, 2737, 671, 271]


In [10]:
bpe_tokenizer.get_vocab_size()

20000

In [1]:
from tokenizers.processors import TemplateProcessing
import numpy as np
import pickle

pad_id = bpe_tokenizer.token_to_id("<pad>")
sos_id = bpe_tokenizer.token_to_id("<s>")
eos_id = bpe_tokenizer.token_to_id("</s>")

bpe_tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[("<s>", sos_id), ("</s>", eos_id)])

bpe_tokenizer.enable_truncation(max_length=128)
bpe_tokenizer.enable_padding(length=128, pad_id=pad_id, pad_token="<pad>")

en_tok = np.array([x.ids for x in bpe_tokenizer.encode_batch(en)], dtype=np.int16)
de_tok = np.array([x.ids for x in bpe_tokenizer.encode_batch(de)], dtype=np.int16)

with open('trans_data.pkl', 'wb') as f:
    pickle.dump((en_tok, de_tok), f)
